### **Accessing Audio Data from Google Drive**

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### **Unzip Audio data**

In [2]:
!unzip -q "/content/drive/MyDrive/THESIS/Audio.zip" -d "/content/audio_data"

### **Verify Unzip Data**

In [3]:
import os
from pathlib import Path

# Update this to match your unzipped location
base_path = Path("/content/audio_data/Audio")

if base_path.exists():
    # 1. Count Subject folders (Case-insensitive check)
    subjects = [d for d in base_path.iterdir() if d.is_dir() and d.name.lower().startswith("subject")]
    print(f"✅ Found {len(subjects)} Subject folders.")

    # 2. Check a sample subject for audio files
    if len(subjects) > 0:
        sample_sub = subjects[0]
        # Check for 'Audio' subfolder inside the subject folder (if your zip had that structure)
        audio_sub = sample_sub / "Audio"
        search_path = audio_sub if audio_sub.exists() else sample_sub

        wav_files = list(search_path.glob("*.wav")) + list(search_path.glob("*.WAV"))
        print(f"📂 Sample Folder: {sample_sub.name}")
        print(f"🎵 Audio files in sample: {len(wav_files)}")

        if len(wav_files) > 0:
            print(f"📄 Example file: {wav_files[0].name}")
else:
    print("❌ Error: Path not found. Please check your unzip destination.")

✅ Found 42 Subject folders.
📂 Sample Folder: subject39
🎵 Audio files in sample: 100
📄 Example file: 038_Trial_02_Speaking_Calmness_aud.wav


### **CSV Data**

In [7]:
import os
import pandas as pd
from pathlib import Path
from tqdm import tqdm

# 1. Define Local Paths
RAW_DATA_PATH = Path("/content/audio_data/Audio")
LOCAL_METADATA_PATH = Path("/content/audio_metadata_local.csv")

# 2. Emotion Mapping
EMOTION_MAPPING = {
    'Happiness': 'H', 'Happy': 'H', 'Angry': 'A', 'Anger': 'A',
    'Calm': 'C', 'Calmness': 'C', 'Neutral': 'N', 'Sad': 'S', 'Sadness': 'S'
}

def create_local_metadata():
    records = []
    # Get all subject folders
    subject_folders = sorted([d for d in RAW_DATA_PATH.iterdir() if d.is_dir()])

    for sub_folder in tqdm(subject_folders, desc="Scanning Folders"):
        audio_dir = sub_folder / "Audio"
        if not audio_dir.exists(): audio_dir = sub_folder

        wav_files = list(audio_dir.glob("*.wav")) + list(audio_dir.glob("*.WAV"))

        for fp in wav_files:
            filename = fp.stem
            # Simple Emotion Detection from filename
            emotion_code = 'N' # Default to Neutral
            for emo, code in EMOTION_MAPPING.items():
                if emo.lower() in filename.lower():
                    emotion_code = code
                    break

            records.append({
                "subject_id": sub_folder.name,
                "emotion_code": emotion_code,
                "filepath": str(fp)
            })

    df = pd.DataFrame(records)
    df.to_csv(LOCAL_METADATA_PATH, index=False)
    print(f"\n✅ Metadata saved LOCALLY at: {LOCAL_METADATA_PATH}")
    print(f"📊 Total Samples Found: {len(df)}")
    return df

# Run it
df_local = create_local_metadata()

Scanning Folders: 100%|██████████| 42/42 [00:00<00:00, 848.66it/s]


✅ Metadata saved LOCALLY at: /content/audio_metadata_local.csv
📊 Total Samples Found: 4200


### **Installing libraries**

In [9]:
!pip install openai-whisper
!pip install transformers accelerate
!pip install librosa h5py

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 16.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803980 sha256=5d7a3ebe020ffa6f3c8a01012f3aad7bfccd2133d5a10ba20b2cccd27e899933
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper


### **Feature Extraction**

In [10]:
import os
import sys
import time
import sqlite3
import logging
import re
import warnings
import shutil
from pathlib import Path
from typing import Tuple

import numpy as np
import pandas as pd
import librosa
import librosa.feature
import h5py
import torch
import whisper
from transformers import (
    Wav2Vec2Processor,
    Wav2Vec2Model,
    AutoTokenizer,
    AutoModelForSequenceClassification
)
from tqdm import tqdm

warnings.filterwarnings("ignore", category=FutureWarning)

# ============================================
# CONFIGURATION (UPDATED FOR COLAB)
# ============================================
class Config:
    # 1. Use the local Colab paths for maximum extraction speed
    METADATA_PATH = Path("/content/audio_metadata_local.csv")
    OUTPUT_DIR = Path("/content/features_multimodal")

    # 2. Your Google Drive path for saving the final file for PyCharm
    DRIVE_SAVE_DIR = Path("/content/drive/MyDrive/THESIS/1 Data/2 Preprocessed/features_multimodal")

    SAMPLE_RATE = 16000
    DURATION = 20
    TRIM_TOP_DB = 25

    WAV2VEC_MODEL = "facebook/wav2vec2-base-960h"
    WHISPER_MODEL = "base"
    SENTIMENT_MODEL = "cardiffnlp/twitter-roberta-base-sentiment"

    N_MFCC = 40
    HOP_LENGTH = 512
    N_FFT = 2048

    # ⚠️ CHANGED TO FALSE FOR YOUR FULL 4,200 DATASET RUN
    TEST_MODE = False

    @classmethod
    def setup_dirs(cls):
        cls.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        (cls.OUTPUT_DIR / "cache").mkdir(exist_ok=True)
        (cls.OUTPUT_DIR / "logs").mkdir(exist_ok=True)
        cls.DRIVE_SAVE_DIR.mkdir(parents=True, exist_ok=True)

# ============================================
# LOGGING
# ============================================
def setup_logger() -> logging.Logger:
    Config.setup_dirs()
    log_file = Config.OUTPUT_DIR / "logs" / f"extraction_{time.strftime('%Y%m%d_%H%M%S')}.log"
    logger = logging.getLogger("FeatureExtractor")
    logger.setLevel(logging.INFO)

    c_handler = logging.StreamHandler(sys.stdout)
    c_handler.setFormatter(logging.Formatter('%(levelname)s - %(message)s'))
    f_handler = logging.FileHandler(log_file, encoding='utf-8')
    f_handler.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(message)s'))

    if not logger.handlers:
        logger.addHandler(c_handler)
        logger.addHandler(f_handler)
    return logger

logger = setup_logger()

# ============================================
# CHECKPOINT MANAGER
# ============================================
class CheckpointManager:
    def __init__(self, db_path: Path):
        self.db_path = db_path
        self._init_db()

    def _init_db(self):
        with sqlite3.connect(self.db_path) as conn:
            conn.execute('CREATE TABLE IF NOT EXISTS processed_files (filepath TEXT PRIMARY KEY, cache_path TEXT, status TEXT)')

    def is_processed(self, filepath: str) -> bool:
        with sqlite3.connect(self.db_path) as conn:
            res = conn.execute('SELECT status FROM processed_files WHERE filepath = ?', (filepath,)).fetchone()
            return res is not None and res[0] == 'SUCCESS'

    def mark_success(self, filepath: str, cache_path: str):
        with sqlite3.connect(self.db_path) as conn:
            conn.execute('INSERT OR REPLACE INTO processed_files VALUES (?, ?, "SUCCESS")', (filepath, cache_path))

# ============================================
# MULTIMODAL FEATURE EXTRACTOR
# ============================================
class MultimodalExtractor:
    def __init__(self):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.checkpoint_db = CheckpointManager(Config.OUTPUT_DIR / "checkpoint.db")
        self.emotion_pattern = re.compile(r'\b(happy|sad|angry|neutral|fear|disgust|anger|calm|calmness)\b', flags=re.IGNORECASE)

    def load_models(self):
        logger.info(f"[INFO] Initializing on Device: {self.device.type.upper()}")
        logger.info("[INFO] Loading Models (Wav2Vec2, Whisper, RoBERTa)...")
        self.w2v_processor = Wav2Vec2Processor.from_pretrained(Config.WAV2VEC_MODEL)
        self.w2v_model = Wav2Vec2Model.from_pretrained(Config.WAV2VEC_MODEL).to(self.device).eval()
        self.whisper_model = whisper.load_model(Config.WHISPER_MODEL, device=self.device)
        self.sent_tokenizer = AutoTokenizer.from_pretrained(Config.SENTIMENT_MODEL)
        self.sent_model = AutoModelForSequenceClassification.from_pretrained(Config.SENTIMENT_MODEL).to(self.device).eval()
        logger.info("[INFO] All Models Loaded Successfully.")

    def extract_traditional(self, audio: np.ndarray) -> np.ndarray:
        noise = np.random.normal(0, 1e-6, len(audio))
        audio_n = audio + noise
        mfcc = librosa.feature.mfcc(y=audio_n, sr=Config.SAMPLE_RATE, n_mfcc=Config.N_MFCC)
        feat = np.concatenate([
            np.mean(mfcc, axis=1), np.std(mfcc, axis=1),
            np.max(mfcc, axis=1), np.min(mfcc, axis=1)
        ])
        return feat

    def extract_wav2vec(self, audio: np.ndarray) -> np.ndarray:
        inputs = self.w2v_processor(audio, sampling_rate=Config.SAMPLE_RATE, return_tensors="pt", padding=True)
        with torch.no_grad():
            outputs = self.w2v_model(inputs.input_values.to(self.device))
            embeddings = outputs.last_hidden_state.mean(dim=1).cpu().numpy().flatten()
        return embeddings

    def extract_semantic(self, audio: np.ndarray) -> Tuple[np.ndarray, str]:
        result = self.whisper_model.transcribe(audio.astype(np.float32), fp16=torch.cuda.is_available())
        text = result["text"].strip()
        if not text: return np.zeros(768), "None"
        scrubbed = self.emotion_pattern.sub('[EMO]', text)
        inputs = self.sent_tokenizer(scrubbed, return_tensors="pt", truncation=True).to(self.device)
        with torch.no_grad():
            outputs = self.sent_model(**inputs, output_hidden_states=True)
            emb = outputs.hidden_states[-1][:, 0, :].cpu().numpy().flatten()
        return emb, scrubbed

    def process_file(self, file_path: str, cache_path: Path) -> bool:
        try:
            y, _ = librosa.load(file_path, sr=Config.SAMPLE_RATE, duration=Config.DURATION)
            yt, _ = librosa.effects.trim(y, top_db=Config.TRIM_TOP_DB)
            if len(yt) < (Config.SAMPLE_RATE * 0.5): yt = y
            audio = librosa.util.normalize(yt)

            trad = self.extract_traditional(audio)
            w2v = self.extract_wav2vec(audio)
            sem, txt = self.extract_semantic(audio)

            np.savez_compressed(cache_path, traditional=trad, wav2vec=w2v, semantic=sem, transcript=txt)
            return True
        except Exception as e:
            logger.error(f"[ERROR] Failed file {file_path}: {e}")
            return False

    def run_pipeline(self, df: pd.DataFrame):
        logger.info(f"[START] Processing {len(df)} files.")
        for idx, row in tqdm(df.iterrows(), total=len(df), desc="Extracting Features"):
            fp = str(row['filepath'])
            subject_clean = str(row['subject_id']).split('.')[0]
            cid = f"subject{subject_clean}_{Path(fp).stem}"
            cp = Config.OUTPUT_DIR / "cache" / f"{cid}.npz"

            if self.checkpoint_db.is_processed(fp) and cp.exists(): continue
            if self.process_file(fp, cp):
                self.checkpoint_db.mark_success(fp, str(cp))
            if idx % 50 == 0: torch.cuda.empty_cache()

    def compile_hdf5(self, df: pd.DataFrame):
        logger.info("[INFO] Aggregating features into Compressed HDF5...")
        hp = Config.OUTPUT_DIR / "multimodal_features.h5"

        with h5py.File(hp, 'w') as hf:
            for _, row in tqdm(df.iterrows(), total=len(df), desc="Compiling HDF5"):
                fp = Path(str(row['filepath']))
                subject_clean = str(row['subject_id']).split('.')[0]
                cid = f"subject{subject_clean}_{fp.stem}"
                cp = Config.OUTPUT_DIR / "cache" / f"{cid}.npz"

                if cp.exists():
                    d = np.load(cp)
                    g = hf.create_group(cid)
                    # ADDED: GZIP Compression (Essential for downloading to PyCharm)
                    g.create_dataset('traditional', data=d['traditional'], compression="gzip")
                    g.create_dataset('wav2vec', data=d['wav2vec'], compression="gzip")
                    g.create_dataset('semantic', data=d['semantic'], compression="gzip")
                    g.attrs['emotion'] = str(row.get('emotion_code', 'UNKNOWN'))
                    g.attrs['subject'] = subject_clean
                    g.attrs['transcript'] = str(d['transcript'])

        logger.info(f"[SUCCESS] HDF5 compiled at: {hp}")
        return hp

# ============================================
# MAIN ORCHESTRATOR
# ============================================
def main():
    # 1. Check Metadata
    if not Config.METADATA_PATH.exists():
        logger.error(f"[FATAL] Metadata CSV not found at {Config.METADATA_PATH}. Did you run the local dataloader?")
        sys.exit(1)

    df = pd.read_csv(Config.METADATA_PATH)

    if Config.TEST_MODE:
        logger.warning("[WARNING] Running in TEST MODE (First 5 files only)")
        df = df.head(5)

    # 2. Extract
    extractor = MultimodalExtractor()
    extractor.load_models()
    extractor.run_pipeline(df)

    # 3. Compile HDF5
    final_local_h5 = extractor.compile_hdf5(df)

    # 4. Save to Google Drive for PyCharm access
    logger.info(f"[INFO] Copying final HDF5 file to Google Drive...")
    drive_dest = Config.DRIVE_SAVE_DIR / "multimodal_features.h5"
    shutil.copy(final_local_h5, drive_dest)

    logger.info("=" * 50)
    logger.info(f"🎉 DONE! File ready for PyCharm at: {drive_dest}")
    logger.info("=" * 50)

if __name__ == "__main__":
    main()

INFO - [INFO] Initializing on Device: CUDA


INFO:FeatureExtractor:[INFO] Initializing on Device: CUDA


INFO - [INFO] Loading Models (Wav2Vec2, Whisper, RoBERTa)...


INFO:FeatureExtractor:[INFO] Loading Models (Wav2Vec2, Whisper, RoBERTa)...
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

...

INFO:FeatureExtractor:==================================================


INFO - 🎉 DONE! File ready for PyCharm at: /content/drive/MyDrive/THESIS/1 Data/2 Preprocessed/features_multimodal/multimodal_features.h5


INFO:FeatureExtractor:🎉 DONE! File ready for PyCharm at: /content/drive/MyDrive/THESIS/1 Data/2 Preprocessed/features_multimodal/multimodal_features.h5


INFO - ==================================================


INFO:FeatureExtractor:==================================================


### **Install Libraries**

In [11]:
!pip install h5py scikit-learn xgboost lightgbm torch


### **Load the HDF5 Feature Dataset**

In [12]:
import h5py
import numpy as np

path = "/content/drive/MyDrive/THESIS/1 Data/2 Preprocessed/features_multimodal/multimodal_features.h5"

file = h5py.File(path, "r")

print(list(file.keys()))

['subjectSubject10_002_Trial_02_Speaking_Neutral_aud', 'subjectSubject10_004_Trial_02_Speaking_Anger_aud', 'subjectSubject10_006_Trial_02_Speaking_Anger_aud', 'subjectSubject10_008_Trial_02_Speaking_Anger_aud', 'subjectSubject10_010_Trial_02_Speaking_Anger_aud', 'subjectSubject10_012_Trial_02_Speaking_Neutral_aud', 'subjectSubject10_014_Trial_02_Speaking_Anger_aud', 'subjectSubject10_016_Trial_02_Speaking_Anger_aud', 'subjectSubject10_018_Trial_02_Speaking_Anger_aud', 'subjectSubject10_020_Trial_02_Speaking_Anger_aud', 'subjectSubject10_022_Trial_02_Speaking_Neutral_aud', 'subjectSubject10_024_Trial_02_Speaking_Calmness_aud', 'subjectSubject10_026_Trial_02_Speaking_Calmness_aud', 'subjectSubject10_028_Trial_02_Speaking_Calmness_aud', 'subjectSubject10_030_Trial_02_Speaking_Calmness_aud', 'subjectSubject10_032_Trial_02_Speaking_Neutral_aud', 'subjectSubject10_034_Trial_02_Speaking_Calmness_aud', 'subjectSubject10_036_Trial_02_Speaking_Calmness_aud', 'subjectSubject10_038_Trial_02_Speaki

### **Model Training**

In [15]:
import h5py
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import random
from tqdm import tqdm
import seaborn as sns
import matplotlib.pyplot as plt

# ==========================================
# 1. CONFIGURATION & SEEDING (For Reproducibility)
# ==========================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
H5_PATH = "/content/drive/MyDrive/THESIS/1 Data/2 Preprocessed/features_multimodal/multimodal_features.h5"
BATCH_SIZE = 64
EPOCHS = 50

# Map your specific emotions to integers
LABEL_MAP = {'H': 0, 'N': 1, 'A': 2, 'S': 3, 'C': 4}
REVERSE_MAP = {v: k for k, v in LABEL_MAP.items()}

# ==========================================
# 2. SUBJECT-INDEPENDENT DATA LOADER
# ==========================================
def load_and_split_data(h5_path):
    print("📦 Loading Data and Performing Subject-Independent Split...")
    data_dict = {}

    with h5py.File(h5_path, 'r') as hf:
        for key in hf.keys():
            group = hf[key]
            subject = group.attrs['subject']
            label = LABEL_MAP.get(group.attrs['emotion'], 1) # Default Neutral if error

            if subject not in data_dict:
                data_dict[subject] = {'trad': [], 'w2v': [], 'sem': [], 'labels': []}

            data_dict[subject]['trad'].append(group['traditional'][:])
            data_dict[subject]['w2v'].append(group['wav2vec'][:])
            data_dict[subject]['sem'].append(group['semantic'][:])
            data_dict[subject]['labels'].append(label)

    subjects = list(data_dict.keys())
    random.shuffle(subjects)

    # 30 Train (71%), 6 Val (14%), 6 Test (14%)
    train_subs = subjects[:30]
    val_subs = subjects[30:36]
    test_subs = subjects[36:]

    def compile_split(sub_list):
        trad, w2v, sem, labels = [], [], [], []
        for sub in sub_list:
            trad.extend(data_dict[sub]['trad'])
            w2v.extend(data_dict[sub]['w2v'])
            sem.extend(data_dict[sub]['sem'])
            labels.extend(data_dict[sub]['labels'])
        return np.array(trad), np.array(w2v), np.array(sem), np.array(labels)

    train_data = compile_split(train_subs)
    val_data = compile_split(val_subs)
    test_data = compile_split(test_subs)

    print(f"✅ Split Complete! Train: {len(train_data[3])} | Val: {len(val_data[3])} | Test: {len(test_data[3])}")
    return train_data, val_data, test_data

# ==========================================
# 3. STRICT NORMALIZATION (Preventing Leakage)
# ==========================================
def scale_features(train_data, val_data, test_data):
    print("⚖️ Normalizing Features (Fitting on Train only)...")
    scalers = {'trad': StandardScaler(), 'w2v': StandardScaler(), 'sem': StandardScaler()}

    # Unpack
    tr_t, tr_w, tr_s, tr_y = train_data
    v_t, v_w, v_s, v_y = val_data
    te_t, te_w, te_s, te_y = test_data

    # Fit and Transform Train
    tr_t = scalers['trad'].fit_transform(tr_t)
    tr_w = scalers['w2v'].fit_transform(tr_w)
    tr_s = scalers['sem'].fit_transform(tr_s)

    # Transform Val and Test ONLY
    v_t = scalers['trad'].transform(v_t)
    v_w = scalers['w2v'].transform(v_w)
    v_s = scalers['sem'].transform(v_s)

    te_t = scalers['trad'].transform(te_t)
    te_w = scalers['w2v'].transform(te_w)
    te_s = scalers['sem'].transform(te_s)

    return (tr_t, tr_w, tr_s, tr_y), (v_t, v_w, v_s, v_y), (te_t, te_w, te_s, te_y)

class EmotionDataset(Dataset):
    def __init__(self, data_tuple):
        self.trad = torch.FloatTensor(data_tuple[0])
        self.w2v = torch.FloatTensor(data_tuple[1])
        self.sem = torch.FloatTensor(data_tuple[2])
        self.labels = torch.LongTensor(data_tuple[3])

    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        return self.trad[idx], self.w2v[idx], self.sem[idx], self.labels[idx]

# ==========================================
# 4. ROBUST MULTIMODAL ATTENTION MODEL
# ==========================================
class AttentionFusionModel(nn.Module):
    def __init__(self, trad_dim=160, w2v_dim=768, sem_dim=768, num_classes=5):
        super(AttentionFusionModel, self).__init__()

        # Feature Extractors with heavy dropout to prevent overfitting
        self.trad_net = nn.Sequential(nn.Linear(trad_dim, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.4))
        self.w2v_net = nn.Sequential(nn.Linear(w2v_dim, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.4))
        self.sem_net = nn.Sequential(nn.Linear(sem_dim, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.4))

        # Attention Mechanism (Learns which modality to trust)
        self.attention = nn.Sequential(nn.Linear(256 * 3, 3), nn.Softmax(dim=1))

        # Final Classifier
        self.classifier = nn.Sequential(
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, trad, w2v, sem):
        # 1. Modality Dropout (During training, randomly drop a modality to ensure robustness)
        if self.training:
            if random.random() < 0.1: trad = torch.zeros_like(trad)
            if random.random() < 0.1: sem = torch.zeros_like(sem)

        h_trad = self.trad_net(trad)
        h_w2v = self.w2v_net(w2v)
        h_sem = self.sem_net(sem)

        # 2. Concat to calculate attention weights
        combined_features = torch.cat([h_trad, h_w2v, h_sem], dim=1)
        attn_weights = self.attention(combined_features)

        # 3. Apply weights to each modality (Weighted Sum)
        fused_vector = (h_trad * attn_weights[:, 0].unsqueeze(1) +
                        h_w2v * attn_weights[:, 1].unsqueeze(1) +
                        h_sem * attn_weights[:, 2].unsqueeze(1))

        return self.classifier(fused_vector)

# ==========================================
# 5. MASTER TRAINING ORCHESTRATOR
# ==========================================
def train_model():
    # Load, Split, Scale
    train_raw, val_raw, test_raw = load_and_split_data(H5_PATH)
    train_s, val_s, test_s = scale_features(train_raw, val_raw, test_raw)

    train_loader = DataLoader(EmotionDataset(train_s), batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(EmotionDataset(val_s), batch_size=BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(EmotionDataset(test_s), batch_size=BATCH_SIZE, shuffle=False)

    model = AttentionFusionModel().to(DEVICE)
    criterion = nn.CrossEntropyLoss()

    # AdamW includes better weight decay to prevent overfitting
    optimizer = optim.AdamW(model.parameters(), lr=0.0005, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)

    best_val_acc = 0
    patience_counter = 0
    early_stop_patience = 8

    print(f"\n🚀 Starting Training on {DEVICE}...")

    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0

        for trad, w2v, sem, labels in train_loader:
            trad, w2v, sem, labels = trad.to(DEVICE), w2v.to(DEVICE), sem.to(DEVICE), labels.to(DEVICE)

            optimizer.zero_grad()
            outputs = model(trad, w2v, sem)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        # Validation
        model.eval()
        val_correct, val_total = 0, 0
        with torch.no_grad():
            for trad, w2v, sem, labels in val_loader:
                trad, w2v, sem, labels = trad.to(DEVICE), w2v.to(DEVICE), sem.to(DEVICE), labels.to(DEVICE)
                outputs = model(trad, w2v, sem)
                _, predicted = torch.max(outputs.data, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

        val_acc = 100 * val_correct / val_total
        print(f"Epoch {epoch+1:02d}/{EPOCHS} | Loss: {train_loss/len(train_loader):.4f} | Val Acc: {val_acc:.2f}%")

        scheduler.step(val_acc)

        # Model Checkpointing & Early Stopping
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), "/content/best_emotion_model.pt")
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= early_stop_patience:
                print("🛑 Early Stopping triggered to prevent overfitting!")
                break

    # ==========================================
    # 6. FINAL TEST EVALUATION
    # ==========================================
    print("\n🏆 Loading Best Model for Final Test Evaluation...")
    model.load_state_dict(torch.load("/content/best_emotion_model.pt"))
    model.eval()

    y_true, y_pred = [], []
    with torch.no_grad():
        for trad, w2v, sem, labels in test_loader:
            trad, w2v, sem = trad.to(DEVICE), w2v.to(DEVICE), sem.to(DEVICE)
            outputs = model(trad, w2v, sem)
            _, predicted = torch.max(outputs.data, 1)
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(predicted.cpu().numpy())

    print("\n📊 FINAL CLASSIFICATION REPORT (Unseen Subjects):")
    target_names = [REVERSE_MAP[i] for i in range(5)]
    print(classification_report(y_true, y_pred, target_names=target_names))

if __name__ == "__main__":
    train_model()

📦 Loading Data and Performing Subject-Independent Split...
✅ Split Complete! Train: 3000 | Val: 600 | Test: 600
⚖️ Normalizing Features (Fitting on Train only)...

🚀 Starting Training on cuda...
Epoch 01/50 | Loss: 0.6440 | Val Acc: 96.33%
Epoch 02/50 | Loss: 0.2752 | Val Acc: 98.50%
Epoch 03/50 | Loss: 0.2228 | Val Acc: 98.67%
Epoch 04/50 | Loss: 0.0935 | Val Acc: 98.67%
Epoch 05/50 | Loss: 0.0646 | Val Acc: 98.67%
Epoch 06/50 | Loss: 0.0780 | Val Acc: 98.83%
Epoch 07/50 | Loss: 0.0971 | Val Acc: 98.83%
Epoch 08/50 | Loss: 0.0952 | Val Acc: 98.50%
Epoch 09/50 | Loss: 0.0477 | Val Acc: 99.00%
Epoch 10/50 | Loss: 0.0912 | Val Acc: 99.33%
Epoch 11/50 | Loss: 0.0575 | Val Acc: 99.00%
Epoch 12/50 | Loss: 0.0667 | Val Acc: 99.00%
Epoch 13/50 | Loss: 0.0530 | Val Acc: 99.00%
Epoch 14/50 | Loss: 0.0708 | Val Acc: 99.17%
Epoch 15/50 | Loss: 0.0898 | Val Acc: 99.33%
Epoch 16/50 | Loss: 0.0239 | Val Acc: 99.00%
Epoch 17/50 | Loss: 0.0428 | Val Acc: 99.00%
Epoch 18/50 | Loss: 0.0499 | Val Acc: 99

### **Download the model training Scalars**

In [17]:
import h5py
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
import random
import pickle
from tqdm import tqdm

# ==========================================
# 1. SETUP & CONFIGURATION
# ==========================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
H5_PATH = "/content/drive/MyDrive/THESIS/1 Data/2 Preprocessed/features_multimodal/multimodal_features.h5"
LABEL_MAP = {'H': 0, 'N': 1, 'A': 2, 'S': 3, 'C': 4}
REVERSE_MAP = {v: k for k, v in LABEL_MAP.items()}

# ==========================================
# 2. MODEL ARCHITECTURE
# ==========================================
class AttentionFusionModel(nn.Module):
    def __init__(self, trad_dim=160, w2v_dim=768, sem_dim=768, num_classes=5):
        super(AttentionFusionModel, self).__init__()
        # Branch Experts
        self.trad_net = nn.Sequential(nn.Linear(trad_dim, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.4))
        self.w2v_net = nn.Sequential(nn.Linear(w2v_dim, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.4))
        self.sem_net = nn.Sequential(nn.Linear(sem_dim, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.4))

        # Self-Attention Logic
        self.attention = nn.Sequential(nn.Linear(256 * 3, 3), nn.Softmax(dim=1))

        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, trad, w2v, sem):
        h_t, h_w, h_s = self.trad_net(trad), self.w2v_net(w2v), self.sem_net(sem)
        attn = self.attention(torch.cat([h_t, h_w, h_s], dim=1))
        fused = (h_t * attn[:, 0].unsqueeze(1) + h_w * attn[:, 1].unsqueeze(1) + h_s * attn[:, 2].unsqueeze(1))
        return self.classifier(fused)

# ==========================================
# 3. DATASET CLASS
# ==========================================
class EmotionDataset(Dataset):
    def __init__(self, t, w, s, l):
        self.t, self.w, self.s, self.l = torch.FloatTensor(t), torch.FloatTensor(w), torch.FloatTensor(s), torch.LongTensor(l)
    def __len__(self): return len(self.l)
    def __getitem__(self, i): return self.t[i], self.w[i], self.s[i], self.l[i]

# ==========================================
# 4. THE MAIN EXECUTION BLOCK
# ==========================================
print("📦 Step 1: Loading Data and Splitting by Subjects...")
data_dict = {}
with h5py.File(H5_PATH, 'r') as hf:
    for key in hf.keys():
        sub = hf[key].attrs['subject']
        if sub not in data_dict: data_dict[sub] = {'t':[], 'w':[], 's':[], 'l':[]}
        data_dict[sub]['t'].append(hf[key]['traditional'][:])
        data_dict[sub]['w'].append(hf[key]['wav2vec'][:])
        data_dict[sub]['s'].append(hf[key]['semantic'][:])
        data_dict[sub]['l'].append(LABEL_MAP.get(hf[key].attrs['emotion'], 1))

subs = list(data_dict.keys())
random.shuffle(subs)
train_subs, val_subs, test_subs = subs[:30], subs[30:36], subs[36:]

def get_data(s_list):
    t, w, s, l = [], [], [], []
    for sub in s_list:
        t.extend(data_dict[sub]['t']); w.extend(data_dict[sub]['w'])
        s.extend(data_dict[sub]['s']); l.extend(data_dict[sub]['l'])
    return np.array(t), np.array(w), np.array(s), np.array(l)

tr_raw, val_raw, te_raw = get_data(train_subs), get_data(val_subs), get_data(test_subs)

print("⚖️ Step 2: Normalizing Features...")
scalers = {'trad': StandardScaler(), 'w2v': StandardScaler(), 'sem': StandardScaler()}
# Fit ONLY on training data
tr_t = scalers['trad'].fit_transform(tr_raw[0])
tr_w = scalers['w2v'].fit_transform(tr_raw[1])
tr_s = scalers['sem'].fit_transform(tr_raw[2])

# Apply to Val/Test
val_t, val_w, val_s = scalers['trad'].transform(val_raw[0]), scalers['w2v'].transform(val_raw[1]), scalers['sem'].transform(val_raw[2])
te_t, te_w, te_s = scalers['trad'].transform(te_raw[0]), scalers['w2v'].transform(te_raw[1]), scalers['sem'].transform(te_raw[2])

# Save Scalers for PyCharm
with open('/content/scalers.pkl', 'wb') as f:
    pickle.dump(scalers, f)
print("✅ Scalers saved to /content/scalers.pkl")

# ==========================================
# 5. TRAINING LOOP
# ==========================================
train_loader = DataLoader(EmotionDataset(tr_t, tr_w, tr_s, tr_raw[3]), batch_size=64, shuffle=True)
val_loader = DataLoader(EmotionDataset(val_t, val_w, val_s, val_raw[3]), batch_size=64)

model = AttentionFusionModel().to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.0005)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)

best_acc = 0
for epoch in range(50):
    model.train()
    for t, w, s, l in train_loader:
        t, w, s, l = t.to(DEVICE), w.to(DEVICE), s.to(DEVICE), l.to(DEVICE)
        optimizer.zero_grad(); loss = criterion(model(t, w, s), l); loss.backward(); optimizer.step()

    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for t, w, s, l in val_loader:
            t, w, s, l = t.to(DEVICE), w.to(DEVICE), s.to(DEVICE), l.to(DEVICE)
            _, pred = torch.max(model(t, w, s), 1)
            total += l.size(0); correct += (pred == l).sum().item()

    acc = 100 * correct / total
    print(f"Epoch {epoch+1} | Val Acc: {acc:.2f}%")
    scheduler.step(acc)
    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), "/content/best_emotion_model.pt")

print(f"🎉 Success! Best Val Acc: {best_acc:.2f}%")
print("👉 DOWNLOAD: 'best_emotion_model.pt' and 'scalers.pkl' from the sidebar.")

📦 Step 1: Loading Data and Splitting by Subjects...
⚖️ Step 2: Normalizing Features...
✅ Scalers saved to /content/scalers.pkl
Epoch 1 | Val Acc: 96.50%
Epoch 2 | Val Acc: 98.67%
Epoch 3 | Val Acc: 98.50%
Epoch 4 | Val Acc: 98.67%
Epoch 5 | Val Acc: 99.00%
Epoch 6 | Val Acc: 99.00%
Epoch 7 | Val Acc: 99.00%
Epoch 8 | Val Acc: 98.67%
Epoch 9 | Val Acc: 98.83%
Epoch 10 | Val Acc: 99.33%
Epoch 11 | Val Acc: 99.00%
Epoch 12 | Val Acc: 98.83%
Epoch 13 | Val Acc: 99.17%
Epoch 14 | Val Acc: 99.17%
Epoch 15 | Val Acc: 99.17%
Epoch 16 | Val Acc: 99.17%
Epoch 17 | Val Acc: 99.17%
Epoch 18 | Val Acc: 98.83%
Epoch 19 | Val Acc: 99.00%
Epoch 20 | Val Acc: 99.17%
Epoch 21 | Val Acc: 99.00%
Epoch 22 | Val Acc: 99.00%
Epoch 23 | Val Acc: 99.17%
Epoch 24 | Val Acc: 99.17%
Epoch 25 | Val Acc: 99.00%
Epoch 26 | Val Acc: 99.00%
Epoch 27 | Val Acc: 99.17%
Epoch 28 | Val Acc: 99.00%
Epoch 29 | Val Acc: 99.17%
Epoch 30 | Val Acc: 99.17%
Epoch 31 | Val Acc: 99.00%
Epoch 32 | Val Acc: 99.17%
Epoch 33 | Val Acc